# EmosiKu: Klasifikasi Indikasi Gangguan Kesehatan Mental pada Teks Media Sosial Berbahasa Indonesia Menggunakan Model IndoBERT

**Sistem:** Deteksi Depresi dan Kecemasan berbasis NLP (Text Classification)
**Model Utama:** IndoBERT


In [ ]:
!pip install transformers datasets Sastrawi scikit-learn pandas numpy torch

## 1. Import Library & Koneksi Google Drive

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import os
from google.colab import drive
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Hubungkan Google Colab dengan Google Drive Anda
drive.mount('/content/drive')

## 2. Load Dataset dari Google Drive
Dataset dibaca langsung dari folder yang ada di Google Drive Anda. Pastikan Anda menyesuaikan `path_folder` di bawah ini dengan lokasi folder data Anda di Drive.

In [ ]:
# Ganti path ini sesuai dengan lokasi folder dataset di Google Drive Anda
# Contoh: '/content/drive/MyDrive/NLP_Skripsi/DATASET/'
path_folder = '/content/drive/MyDrive/1SlRZbKGTxz8_DKWj-QLZppwBkMidpNMu/' # Anda bisa menyesuaikan ini jika Anda menambahkan folder tersebut ke Drive sebagai Shortcut

# Atau jika Anda meng-upload foldernya secara manual ke Drive Anda:
# path_folder = '/content/drive/MyDrive/EmosiKu/DATASET/'

try:
    train_df = pd.read_csv(os.path.join(path_folder, 'datd_train.csv'))
    test_df = pd.read_csv(os.path.join(path_folder, 'datd_test.csv'))
    print("Dataset berhasil dimuat!")
    display(train_df.head())
except Exception as e:
    print(f"Error memuat dataset: {e}")
    print("Pastikan path_folder sudah benar dan mengarah ke folder yang berisi datd_train.csv")

## 3. Preprocessing Data

In [ ]:
factory_stopword = StopWordRemoverFactory()
stopword = factory_stopword.create_stop_word_remover()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    text = stopword.remove(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean_text'] = train_df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)

display(train_df[['text', 'clean_text', 'label']].head())

## 4. Tokenisasi dengan IndoBERT

In [ ]:
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_df['clean_text'].tolist(), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_df['clean_text'].tolist(), truncation=True, padding=True, max_length=128)

In [ ]:
class MentalHealthDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MentalHealthDataset(train_encodings, train_df['label'].tolist())
test_dataset = MentalHealthDataset(test_encodings, test_df['label'].tolist())

## 5. Fine-Tuning Model IndoBERT

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

training_args = TrainingArguments(
    output_dir='./EmosiKu_Model',          
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=64,   
    warmup_steps=500,                
    weight_decay=0.01,               
    logging_dir='./logs',            
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,                         
    args=training_args,                  
    train_dataset=train_dataset,         
    eval_dataset=test_dataset,           
    compute_metrics=compute_metrics,
)

## 6. Training & Evaluasi Model

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(f"Hasil Evaluasi: {eval_results}")

In [ ]:
# Simpan model terbaik ke folder Google Drive agar tidak hilang saat Colab ditutup
model_save_path = '/content/drive/MyDrive/EmosiKu_Model/best_model'
os.makedirs(model_save_path, exist_ok=True)
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model berhasil disimpan secara permanen di {model_save_path}")